# CordiaAI — Step 17: Cardiac MRI branch (ACDC + Swin-UNETR) → measurements, diagnosis, 3D heart

1. **Segment** the right ventricle, LV myocardium and LV cavity on every short-axis slice with a 2D
   **Swin-UNETR** (transformer encoder + U-Net decoder).
2. **Measure** from the masks: LV/RV volumes, ejection fractions, LV mass, wall thickness
   (`src/imaging/cardiac_metrics.py`).
3. **Diagnose** one of ACDC's 5 groups (Normal, previous MI, dilated CM, hypertrophic CM, abnormal RV)
   from those measurements with a random forest, the approach of the ACDC challenge's top methods.
4. **Build 3D meshes** of the patient's own chambers for the digital twin (`src/imaging/mesh.py`).

**Data:** downloaded automatically from the Hugging Face mirror of ACDC (CC BY-NC-SA 4.0; cite
Bernard et al., IEEE TMI 2018). **Runtime:** T4 GPU.

**Split:** the 100 labelled training patients are split by patient, stratified by group:
70 train / 10 validation / 20 test. If the mirror's `test/` patients also have ground truth and a group
label, they are evaluated as a second, fully external test set.

**Same code as the app:** the cells marked `%%writefile` are copies of this project's own modules
(`src/models/imaging/`, `src/imaging/`), generated from them, so the model trained here is exactly
the one `mobile_app/backend/main.py` loads.

In [ ]:
!pip install -q monai einops nibabel huggingface_hub scikit-image
import torch
print('CUDA:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import os
for d in ['src/models/imaging', 'src/imaging']:
    os.makedirs(d, exist_ok=True)
for f in ['src/__init__.py', 'src/models/__init__.py', 'src/models/imaging/__init__.py', 'src/imaging/__init__.py']:
    open(f, 'a').close()

In [ ]:
%%writefile src/models/imaging/cardiac_seg.py
"""MRI and CT branches: Swin-UNETR segmentation (transformer encoder + U-Net decoder).

A plain ViT gives one label per image; the digital twin needs the outline of
each heart chamber, pixel by pixel. Swin-UNETR (Hatamizadeh et al., 2022)
keeps a transformer as the encoder and adds a convolutional decoder that
produces a full-resolution mask, so it is still a vision-transformer model.
Implementation from MONAI.

- MRI (ACDC): 2D, one short-axis slice at a time. ACDC slices are 5-10 mm
  apart, so per-slice 2D segmentation is the standard choice. Labels follow
  ACDC: 1 = RV cavity, 2 = LV myocardium, 3 = LV cavity.
  Trained in `notebooks/step17_mri_acdc_colab.ipynb`.
- CT (ImageCAS): 3D patches, binary coronary artery mask.
  Trained in `notebooks/step18_ct_imagecas_colab.ipynb`.
"""
from __future__ import annotations

import inspect

from monai.networks.nets import SwinUNETR

ACDC_LABELS = {0: "background", 1: "RV cavity", 2: "LV myocardium", 3: "LV cavity"}
ACDC_GROUPS = ["NOR", "MINF", "DCM", "HCM", "RV"]
ACDC_GROUP_NAMES = {
    "NOR": "Normal",
    "MINF": "Previous myocardial infarction",
    "DCM": "Dilated cardiomyopathy",
    "HCM": "Hypertrophic cardiomyopathy",
    "RV": "Abnormal right ventricle",
}

MRI_SLICE_SIZE = 224  # divisible by 32, as Swin-UNETR requires
CT_PATCH_SIZE = (96, 96, 96)


def _swin_unetr(img_size, **kwargs) -> SwinUNETR:
    # MONAI deprecated and then removed the img_size argument; pass it only
    # to versions that still take it.
    if "img_size" in inspect.signature(SwinUNETR.__init__).parameters:
        kwargs["img_size"] = img_size
    return SwinUNETR(**kwargs)


def build_mri_segmenter(feature_size: int = 24) -> SwinUNETR:
    return _swin_unetr(
        (MRI_SLICE_SIZE, MRI_SLICE_SIZE), in_channels=1, out_channels=len(ACDC_LABELS),
        feature_size=feature_size, spatial_dims=2,
    )


def build_ct_segmenter(feature_size: int = 48, use_checkpoint: bool = True) -> SwinUNETR:
    return _swin_unetr(
        CT_PATCH_SIZE, in_channels=1, out_channels=2, feature_size=feature_size,
        spatial_dims=3, use_checkpoint=use_checkpoint,
    )

In [ ]:
%%writefile src/imaging/preprocess.py
"""Image preprocessing shared by training notebooks and the app.

Keeping one copy of each conversion matters for the same reason it did for
the ECG spectrogram (see gradcam_ecg.py): the app must feed a model exactly
what it was trained on, or predictions silently degrade.
"""
from __future__ import annotations

import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

VIT_SIZE = 224


def _vit_normalise(x: np.ndarray) -> np.ndarray:
    # google/vit-base-patch16-224-in21k's preprocessor: scale to [0,1], then mean 0.5 / std 0.5
    return (x.astype(np.float32) / 255.0 - 0.5) / 0.5


# ---------------------------------------------------------------- X-ray
def xray_to_tensor(image: Image.Image) -> torch.Tensor:
    """Any chest X-ray image (PNG/JPG, grey or colour) -> (3, 224, 224)."""
    grey = image.convert("L").resize((VIT_SIZE, VIT_SIZE), Image.BILINEAR)
    arr = np.stack([np.array(grey)] * 3, axis=-1)
    return torch.from_numpy(_vit_normalise(arr)).permute(2, 0, 1)


# ---------------------------------------------------------------- Echo
def read_video_frames(path: str | Path) -> np.ndarray:
    """All frames of a video file as (frames, H, W) uint8 grey."""
    import cv2

    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()
    if not frames:
        raise ValueError(f"No frames could be read from {path}")
    return np.stack(frames)


def read_video_bytes(data: bytes, suffix: str = ".avi") -> np.ndarray:
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / f"upload{suffix}"
        path.write_bytes(data)
        return read_video_frames(path)


def sample_clip(frames: np.ndarray, n_frames: int, stride: int = 2, start: int | None = None) -> np.ndarray:
    """n_frames frames, `stride` apart, starting at `start` (default: centred).
    Short videos are looped, as in the EchoNet-Dynamic reference code."""
    span = n_frames * stride
    if start is None:
        start = max(0, (len(frames) - span) // 2)
    idx = (start + np.arange(n_frames) * stride) % len(frames)
    return frames[idx]


def clip_to_tensor(clip: np.ndarray) -> torch.Tensor:
    """(frames, H, W) uint8 -> (frames, 3, 224, 224) normalised for the ViT."""
    t = torch.from_numpy(clip.astype(np.float32)).unsqueeze(1)
    t = F.interpolate(t, size=(VIT_SIZE, VIT_SIZE), mode="bilinear", align_corners=False)
    t = (t / 255.0 - 0.5) / 0.5
    return t.expand(-1, 3, -1, -1).contiguous()


# ---------------------------------------------------------------- MRI
# (CT has its own resampling pipeline in src/imaging/ct.py)
def normalise_volume(volume: np.ndarray, lo_pct: float = 1, hi_pct: float = 99) -> np.ndarray:
    """Percentile clip then scale to [0, 1] -- MRI intensities have no fixed unit."""
    lo, hi = np.percentile(volume, [lo_pct, hi_pct])
    return np.clip((volume.astype(np.float32) - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)


def slices_to_tensor(volume01: np.ndarray, size: int = VIT_SIZE) -> torch.Tensor:
    """(H, W, slices) in [0,1] -> (slices, 1, size, size)."""
    t = torch.from_numpy(np.ascontiguousarray(np.moveaxis(volume01, -1, 0))).unsqueeze(1)
    return F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)


@torch.no_grad()
def segment_mri_volume(model: torch.nn.Module, volume: np.ndarray, batch: int = 8) -> np.ndarray:
    """Short-axis MRI volume (H, W, slices) -> label mask of the same shape."""
    h, w, _ = volume.shape
    x = slices_to_tensor(normalise_volume(volume))
    labels = []
    for i in range(0, len(x), batch):
        logits = model(x[i : i + batch])
        logits = F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)
        labels.append(logits.argmax(1))
    return np.moveaxis(torch.cat(labels).numpy().astype(np.uint8), 0, -1)


def mask_slices_to_tensor(mask: np.ndarray, size: int = VIT_SIZE) -> torch.Tensor:
    """(H, W, slices) integer labels -> (slices, size, size), nearest-neighbour so labels stay whole."""
    t = torch.from_numpy(np.ascontiguousarray(np.moveaxis(mask, -1, 0))).unsqueeze(1).float()
    return F.interpolate(t, size=(size, size), mode="nearest")[:, 0].long()

In [ ]:
%%writefile src/imaging/cardiac_metrics.py
"""Clinical measurements from segmentation masks.

These are the numbers a cardiologist reads off an MRI or echo report, and
the numbers the digital twin is fitted to: chamber volumes, ejection
fraction, myocardial mass and wall thickness. All formulas are standard:

- Volume (MRI): voxel count x voxel volume (Simpson's method of discs over
  the short-axis stack, which is what summing voxels slice by slice is).
- Ejection fraction: (EDV - ESV) / EDV.
- LV mass: myocardial volume x 1.05 g/ml (myocardial tissue density).
- Wall thickness: twice the largest inscribed-circle radius of the
  myocardium on any slice (distance transform), an upper-bound estimate.
- Volume (2D echo): single-plane area-length, V = 8 A^2 / (3 pi L).
- Body surface area: Mosteller, sqrt(height_cm x weight_kg / 3600).
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage

RV, MYO, LV = 1, 2, 3  # ACDC label values
MYOCARDIUM_DENSITY = 1.05  # g/ml


def volume_ml(mask: np.ndarray, label: int, spacing_mm) -> float:
    return float((mask == label).sum() * np.prod(spacing_mm) / 1000.0)


def ejection_fraction(edv: float, esv: float) -> float:
    return float(100.0 * (edv - esv) / edv) if edv > 0 else float("nan")


def max_wall_thickness_mm(mask: np.ndarray, spacing_mm, label: int = MYO) -> float:
    """mask: (H, W, slices). In-plane distance transform on each slice."""
    best = 0.0
    for k in range(mask.shape[-1]):
        myo = mask[..., k] == label
        if myo.any():
            dist = ndimage.distance_transform_edt(myo, sampling=spacing_mm[:2])
            best = max(best, float(dist.max()))
    return 2.0 * best


def bsa_mosteller(height_cm: float, weight_kg: float) -> float:
    return float(np.sqrt(height_cm * weight_kg / 3600.0))


def mri_function(ed_mask: np.ndarray, es_mask: np.ndarray, spacing_mm,
                 height_cm: float | None = None, weight_kg: float | None = None) -> dict:
    """Both ventricles' volumes and EF, LV mass and wall thickness from ED + ES masks."""
    lv_edv, lv_esv = volume_ml(ed_mask, LV, spacing_mm), volume_ml(es_mask, LV, spacing_mm)
    rv_edv, rv_esv = volume_ml(ed_mask, RV, spacing_mm), volume_ml(es_mask, RV, spacing_mm)
    myo_ed = volume_ml(ed_mask, MYO, spacing_mm)
    out = {
        "lv_edv_ml": lv_edv,
        "lv_esv_ml": lv_esv,
        "lv_sv_ml": lv_edv - lv_esv,
        "lv_ef_percent": ejection_fraction(lv_edv, lv_esv),
        "rv_edv_ml": rv_edv,
        "rv_esv_ml": rv_esv,
        "rv_ef_percent": ejection_fraction(rv_edv, rv_esv),
        "lv_mass_g": myo_ed * MYOCARDIUM_DENSITY,
        "myo_es_ml": volume_ml(es_mask, MYO, spacing_mm),
        "max_wall_thickness_mm": max_wall_thickness_mm(ed_mask, spacing_mm),
        "rv_lv_edv_ratio": rv_edv / lv_edv if lv_edv > 0 else float("nan"),
        "mass_edv_ratio": myo_ed * MYOCARDIUM_DENSITY / lv_edv if lv_edv > 0 else float("nan"),
    }
    if height_cm and weight_kg:
        bsa = bsa_mosteller(height_cm, weight_kg)
        out["bsa_m2"] = bsa
        for key in ("lv_edv_ml", "lv_esv_ml", "rv_edv_ml", "rv_esv_ml", "lv_mass_g"):
            out[f"{key}_per_m2"] = out[key] / bsa
    return out


# Features for the ACDC 5-group diagnosis classifier, in a fixed order.
# Indexed volumes need height + weight, which ACDC provides for every patient.
ACDC_FEATURES = [
    "lv_edv_ml_per_m2", "lv_esv_ml_per_m2", "lv_ef_percent",
    "rv_edv_ml_per_m2", "rv_esv_ml_per_m2", "rv_ef_percent",
    "lv_mass_g_per_m2", "max_wall_thickness_mm", "rv_lv_edv_ratio", "mass_edv_ratio",
]


def acdc_feature_vector(metrics: dict) -> np.ndarray:
    return np.array([metrics[k] for k in ACDC_FEATURES], dtype=np.float32)


# ---------------------------------------------------------------- 2D echo
def long_axis_length_mm(mask2d: np.ndarray, spacing_mm) -> float:
    """Extent of the cavity along its principal axis (PCA of pixel coordinates)."""
    ys, xs = np.nonzero(mask2d)
    if len(xs) < 3:
        return 0.0
    pts = np.stack([ys * spacing_mm[0], xs * spacing_mm[1]], axis=1)
    pts -= pts.mean(0)
    axis = np.linalg.svd(pts, full_matrices=False)[2][0]
    proj = pts @ axis
    return float(proj.max() - proj.min())


def echo_area_length_volume_ml(mask2d: np.ndarray, spacing_mm, label: int = 1) -> float:
    cavity = mask2d == label
    area_mm2 = cavity.sum() * spacing_mm[0] * spacing_mm[1]
    length = long_axis_length_mm(cavity, spacing_mm)
    if length == 0:
        return 0.0
    return float(8 * area_mm2**2 / (3 * np.pi * length) / 1000.0)


def echo_function(ed_mask2d: np.ndarray, es_mask2d: np.ndarray, spacing_mm, label: int = 1) -> dict:
    edv = echo_area_length_volume_ml(ed_mask2d, spacing_mm, label)
    esv = echo_area_length_volume_ml(es_mask2d, spacing_mm, label)
    return {"lv_edv_ml": edv, "lv_esv_ml": esv, "lv_sv_ml": edv - esv, "lv_ef_percent": ejection_fraction(edv, esv)}

In [ ]:
%%writefile src/imaging/mesh.py
"""3D meshes of heart structures from segmentation masks, for the digital twin.

Marching cubes turns each labelled structure (LV cavity, myocardium, RV,
coronary tree) into a triangle surface in millimetres. The result is sent
to heart_digital_twin_3d/ as plain JSON (vertex + face arrays a Three.js
BufferGeometry takes directly), or saved as .obj for any 3D viewer.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage
from skimage import measure


def mask_to_mesh(mask: np.ndarray, spacing_mm, step: int = 2, smooth_sigma: float = 1.0) -> dict | None:
    """Binary mask (H, W, D) -> {"vertices": [[x,y,z]...], "faces": [[i,j,k]...]} in mm."""
    if mask.sum() == 0:
        return None
    # Pad so surfaces touching the volume edge still close, then smooth the
    # blocky voxel edges (MRI slices are 5-10 mm thick).
    padded = np.pad(mask.astype(np.float32), 1)
    if smooth_sigma > 0:
        sigma = [smooth_sigma / s for s in spacing_mm]
        smoothed = ndimage.gaussian_filter(padded, sigma=sigma)
        if smoothed.max() > 0.5:  # a structure only a few voxels across can blur away entirely
            padded = smoothed
    try:
        verts, faces, _, _ = measure.marching_cubes(padded, level=0.5, spacing=tuple(spacing_mm), step_size=step)
    except (RuntimeError, ValueError):  # too small or thin to form a surface
        return None
    verts -= np.array(spacing_mm)  # undo the 1-voxel pad
    return {"vertices": np.round(verts, 2).tolist(), "faces": faces.astype(int).tolist()}


def label_meshes(mask: np.ndarray, spacing_mm, labels: dict[int, str], **kwargs) -> dict[str, dict]:
    meshes = {}
    for value, name in labels.items():
        if value == 0:
            continue
        mesh = mask_to_mesh(mask == value, spacing_mm, **kwargs)
        if mesh is not None:
            meshes[name] = mesh
    return meshes


def mesh_to_obj(mesh: dict) -> str:
    lines = [f"v {x} {y} {z}" for x, y, z in mesh["vertices"]]
    lines += [f"f {a + 1} {b + 1} {c + 1}" for a, b, c in mesh["faces"]]
    return "\n".join(lines) + "\n"

## Download ACDC and read each patient's ED/ES frames, group, height and weight

In [ ]:
import glob, os, re
import nibabel as nib
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download

ACDC = snapshot_download('mathpluscode/ACDC', repo_type='dataset', local_dir='/content/acdc')

def read_info(folder):
    cfg = os.path.join(folder, 'Info.cfg')
    if os.path.exists(cfg):
        kv = dict(line.split(':', 1) for line in open(cfg) if ':' in line)
        kv = {k.strip(): v.strip() for k, v in kv.items()}
        return {'ed_frame': int(kv['ED']), 'es_frame': int(kv['ES']), 'group': kv.get('Group'),
                'height_cm': float(kv['Height']), 'weight_kg': float(kv['Weight'])}
    return None

meta = pd.concat([pd.read_csv(p) for p in glob.glob(f'{ACDC}/*.csv')], ignore_index=True).drop_duplicates('pid')
meta = meta.set_index('pid')

patients, skipped = [], []
for folder in sorted(glob.glob(f'{ACDC}/*/patient*')):
    pid = os.path.basename(folder)
    info = read_info(folder)
    if info is None and pid in meta.index:
        r = meta.loc[pid]
        info = {'ed_frame': int(r['ed_frame']), 'es_frame': int(r['es_frame']), 'group': r.get('pathology'),
                'height_cm': float(r['height']) * (100 if float(r['height']) < 3 else 1), 'weight_kg': float(r['weight'])}
    # The Hugging Face mirror names files patientXXX_sax_ed.nii.gz / _sax_es; the official
    # ACDC release names them patientXXX_frameYY.nii.gz with ED/ES frame numbers in Info.cfg.
    ed, es = (os.path.join(folder, f'{pid}_sax_{k}.nii.gz') for k in ('ed', 'es'))
    if not (os.path.exists(ed) and os.path.exists(es)) and info:
        ed, es = (os.path.join(folder, f"{pid}_frame{info[k]:02d}.nii.gz") for k in ('ed_frame', 'es_frame'))
    if not (info and os.path.exists(ed) and os.path.exists(es)):
        skipped.append(pid)
        continue
    has_gt = os.path.exists(ed.replace('.nii.gz', '_gt.nii.gz')) and os.path.exists(es.replace('.nii.gz', '_gt.nii.gz'))
    patients.append({'pid': pid, 'split_folder': os.path.basename(os.path.dirname(folder)), 'ed': ed, 'es': es, 'has_gt': has_gt, **info})
if skipped:
    print(f'{len(skipped)} patient folders skipped (no ED/ES images or no info), e.g. {skipped[:3]}')
patients = pd.DataFrame(patients)
assert len(patients), f'No ACDC patients found under {ACDC}: check the folder layout ({os.listdir(ACDC)})'
print(patients.groupby(['split_folder', 'has_gt']).size())
print(patients[patients.split_folder == 'train']['group'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
labelled = patients[(patients.split_folder == 'train') & patients.has_gt]
tr, te = train_test_split(labelled, test_size=20, stratify=labelled['group'], random_state=42)
tr, va = train_test_split(tr, test_size=10, stratify=tr['group'], random_state=42)
external = patients[(patients.split_folder != 'train') & patients.has_gt & patients['group'].notna()]
print(f'train {len(tr)}  val {len(va)}  test {len(te)}  external test {len(external)}')

def load(path):
    img = nib.load(path)
    return np.asarray(img.dataobj), img.header.get_zooms()[:3]

## 2D slice dataset (ED and ES frames of every training patient)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from src.imaging.preprocess import normalise_volume, slices_to_tensor, mask_slices_to_tensor

def slice_tensors(frame):
    xs, ys = [], []
    for _, p in frame.iterrows():
        for key in ('ed', 'es'):
            vol, _ = load(p[key]); gt, _ = load(p[key].replace('.nii.gz', '_gt.nii.gz'))
            xs.append(slices_to_tensor(normalise_volume(vol))); ys.append(mask_slices_to_tensor(gt))
    return torch.cat(xs), torch.cat(ys)

x_tr, y_tr = slice_tensors(tr)
print('train slices', x_tr.shape)
train_loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=16, shuffle=True)

## Train Swin-UNETR (Dice + cross-entropy)

In [ ]:
from monai.losses import DiceCELoss
from src.models.imaging.cardiac_seg import build_mri_segmenter, ACDC_LABELS

model = build_mri_segmenter().to(device)
criterion = DiceCELoss(to_onehot_y=True, softmax=True)
optimiser = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
EPOCHS = 80
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, EPOCHS)

def dice(a, b):
    return 2 * (a & b).sum() / (a.sum() + b.sum() + 1e-8)

def predict_mask(path):
    vol, sp = load(path)
    model.eval()
    with torch.no_grad():
        x = slices_to_tensor(normalise_volume(vol)).to(device)
        logits = torch.nn.functional.interpolate(model(x), size=vol.shape[:2], mode='bilinear')
    return np.moveaxis(logits.argmax(1).cpu().numpy(), 0, -1).astype(np.uint8), sp

def volume_dice(frame):
    scores = {k: [] for k in (1, 2, 3)}
    for _, p in frame.iterrows():
        for key in ('ed', 'es'):
            pred, _ = predict_mask(p[key]); gt, _ = load(p[key].replace('.nii.gz', '_gt.nii.gz'))
            for k in scores:
                scores[k].append(dice(pred == k, gt == k))
    return {ACDC_LABELS[k]: float(np.mean(v)) for k, v in scores.items()}

best, best_state = 0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        # light augmentation: random intensity scale per slice
        xb = xb * (0.9 + 0.2 * torch.rand(len(xb), 1, 1, 1, device=device))
        loss = criterion(model(xb), yb.unsqueeze(1))
        optimiser.zero_grad(); loss.backward(); optimiser.step()
    scheduler.step()
    if epoch % 10 == 0:
        d = volume_dice(va); m = np.mean(list(d.values()))
        print(f'epoch {epoch} loss {loss.item():.3f} val Dice', {k: round(v, 3) for k, v in d.items()})
        if best_state is None or m > best:
            best, best_state = m, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
print('Test Dice:', {k: round(v, 3) for k, v in volume_dice(te).items()})

## Measurements: predicted masks vs. ground truth on test patients

In [ ]:
from src.imaging.cardiac_metrics import mri_function, acdc_feature_vector, ACDC_FEATURES

def metrics_for(p, use_gt):
    if use_gt:
        ed, sp = load(p['ed'].replace('.nii.gz', '_gt.nii.gz')); es, _ = load(p['es'].replace('.nii.gz', '_gt.nii.gz'))
    else:
        (ed, sp), (es, _) = predict_mask(p['ed']), predict_mask(p['es'])
    return mri_function(ed, es, sp, p['height_cm'], p['weight_kg'])

rows = []
for _, p in te.iterrows():
    g, m = metrics_for(p, True), metrics_for(p, False)
    rows.append({'pid': p['pid'], 'LV EF true': g['lv_ef_percent'], 'LV EF pred': m['lv_ef_percent'],
                 'RV EF true': g['rv_ef_percent'], 'RV EF pred': m['rv_ef_percent'],
                 'LV EDV true': g['lv_edv_ml'], 'LV EDV pred': m['lv_edv_ml']})
res = pd.DataFrame(rows)
for k in ('LV EF', 'RV EF', 'LV EDV'):
    print(f'{k}: mean absolute error {np.abs(res[k + " true"] - res[k + " pred"]).mean():.1f}')
res.round(1)

## Diagnosis: 5 ACDC groups from the measurements
Trained on ground-truth measurements of the 80 train+val patients; tested end-to-end on the 20 test patients' **predicted** masks, i.e. exactly what the app would do. 20 patients is small, so the 5-fold cross-validation over all 100 patients is printed too.

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score
from src.models.imaging.cardiac_seg import ACDC_GROUPS

trva = pd.concat([tr, va])
X_trva = np.stack([acdc_feature_vector(metrics_for(p, True)) for _, p in trva.iterrows()])
y_trva = trva['group'].map(ACDC_GROUPS.index).values
X_te = np.stack([acdc_feature_vector(metrics_for(p, False)) for _, p in te.iterrows()])
y_te = te['group'].map(ACDC_GROUPS.index).values

clf = RandomForestClassifier(n_estimators=500, random_state=42).fit(X_trva, y_trva)
pred = clf.predict(X_te)
print(f'End-to-end test accuracy (20 patients): {accuracy_score(y_te, pred):.3f}')
print(classification_report(y_te, pred, labels=range(5), target_names=ACDC_GROUPS, zero_division=0))

X_all = np.concatenate([X_trva, np.stack([acdc_feature_vector(metrics_for(p, True)) for _, p in te.iterrows()])])
y_all = np.concatenate([y_trva, y_te])
cv = cross_val_score(RandomForestClassifier(n_estimators=500, random_state=42), X_all, y_all, cv=5)
print(f'5-fold CV accuracy on ground-truth measurements ({len(y_all)} patients): {cv.mean():.3f} +/- {cv.std():.3f}')
print('Feature importance:', dict(sorted(zip(ACDC_FEATURES, [round(float(v), 3) for v in clf.feature_importances_]), key=lambda kv: -kv[1])))
joblib.dump({'model': clf, 'features': ACDC_FEATURES, 'groups': ACDC_GROUPS}, 'acdc_diagnosis.joblib')

## External test: ACDC's own 50 test patients
Never used for training, model selection or the diagnosis classifier: the most honest numbers to report.

In [ ]:
ext = external[external['group'].isin(ACDC_GROUPS)]
if len(ext):
    print(f'External test Dice ({len(ext)} patients):', {k: round(v, 3) for k, v in volume_dice(ext).items()})
    rows = []
    for _, p in ext.iterrows():
        g, m = metrics_for(p, True), metrics_for(p, False)
        rows.append({k: abs(g[k] - m[k]) for k in ('lv_ef_percent', 'rv_ef_percent', 'lv_edv_ml', 'lv_mass_g')})
    print('External mean absolute error:', {k: round(float(v), 1) for k, v in pd.DataFrame(rows).mean().items()})
    X_ext = np.stack([acdc_feature_vector(metrics_for(p, False)) for _, p in ext.iterrows()])
    y_ext = ext['group'].map(ACDC_GROUPS.index).values
    pred_ext = clf.predict(X_ext)
    print(f'External end-to-end diagnosis accuracy: {accuracy_score(y_ext, pred_ext):.3f}')
    print(classification_report(y_ext, pred_ext, labels=range(5), target_names=ACDC_GROUPS, zero_division=0))
else:
    print('No labelled external test patients in this copy of ACDC.')

## 3D heart meshes for the digital twin (one test patient, end-diastole)

In [ ]:
import json
from src.imaging.mesh import label_meshes
p = te.iloc[0]
mask, sp = predict_mask(p['ed'])
meshes = label_meshes(mask, sp, ACDC_LABELS)
print({k: (len(v['vertices']), 'vertices') for k, v in meshes.items()})
json.dump({'pid': p['pid'], 'spacing_mm': [float(s) for s in sp], 'meshes': meshes}, open('sample_heart_mesh.json', 'w'))

import matplotlib.pyplot as plt
fig = plt.figure(figsize=(6, 6)); ax = fig.add_subplot(projection='3d')
for (name, m), c in zip(meshes.items(), ['tab:blue', 'tab:red', 'tab:orange']):
    v, f = np.array(m['vertices']), np.array(m['faces'])
    ax.plot_trisurf(v[:, 0], v[:, 1], f, v[:, 2], color=c, alpha=0.5, label=name)
ax.set_title(f"{p['pid']} ({p['group']}), end-diastole"); plt.show()

In [ ]:
torch.save(best_state, 'mri_swin_unetr.pt')
from google.colab import files
for f in ('mri_swin_unetr.pt', 'acdc_diagnosis.joblib', 'sample_heart_mesh.json'):
    files.download(f)  # put them in cordia_ai/checkpoints/